In [2]:
import json
from pathlib import Path
import jax
import numpy as np
from scipy.io import loadmat
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
from flax import serialization as sz
import pandas as pd
from numpyro.infer import NUTS, MCMC, init_to_median
import jax.random as jr
import numpyro
import numpyro.distributions as dist

from model import (
    MiniGPJax,
    # kernel / mean / likelihood classes
    RBF, Matern32, GaussianLikelihood, ZeroMean, GPParams, Standardizer,
    # state (de)serializers defined in your file
    _kernel_to_state, _kernel_from_state,
    _lik_to_state, _lik_from_state,
    _zeromean_to_state, _zeromean_from_state,
    _gpparams_to_state, _gpparams_from_state,
    _std_to_state, _std_from_state,
)
from model import MiniGPJax, _register_all_serializers_once
_register_all_serializers_once()



## Uncomment the following if data was created again

In [16]:
# import numpy as np
# import pandas as pd
# from scipy.io import loadmat
# from pathlib import Path

# # Load .mat file
# mat = loadmat("Surrogate_sensitivity_across_flowRates/P_flowrate_3.mat")

# X = mat['data_set']
# Y = mat['P_Top_Array']

# # Ensure 2D arrays
# X = np.atleast_2d(X)
# Y = np.atleast_2d(Y)

# # If Y is a column vector stored as (N, 1) or (1, N), fix orientation
# if Y.shape[0] != X.shape[0]:
#     Y = Y.T

# # Concatenate columns
# XY = np.hstack([X, Y])

# # Create column names
# x_cols = [f"x{i}" for i in range(X.shape[1])]
# y_cols = [f"y{i}" for i in range(Y.shape[1])]
# cols = x_cols + y_cols

# # Save to CSV
# out_path = Path("Surrogate_sensitivity_across_flowRates") / "P_flowrate_3_XY.csv"
# pd.DataFrame(XY, columns=cols).to_csv(out_path, index=False)

# print(f"Saved CSV to: {out_path}")


Saved CSV to: Surrogate_sensitivity_across_flowRates/P_flowrate_3_XY.csv


## Start bulilding the surrogate now

In [21]:
TRAIN_CSV = "Surrogate_sensitivity_across_flowRates/P_flowrate_1_XY.csv" #Completed_train_set_10_30_25.csv#"Train_Set_2.csv"
TEST_CSV  = "Completed_Test_set_3.csv"#"Test_Set_2.csv"
CKPT_DIR  = "ckpts/gp_run_sens_01"
dim = 4
gp = MiniGPJax(
    dim=dim,
    kernel="rbf",            # or "matern32" I dont have the others
    fixed_noise=None,        # consider None to learn noise for better generalization
    standardize_x=True,
    standardize_y=True,
    log_y=True,              # ensures: log -> standardize (train stats) -> fit
    jitter=1e-6,
)
gp.load(TRAIN_CSV, TEST_CSV)
gp.fit(lbfgs_max_iter=30, lbfgs_tol=1e-7, num_restarts=2, seed=200)

[ok] Loaded train: X=(1024, 4), y=(1024,) (log→std)
[ok] Loaded test:  X=(256, 4), y=(256,) (log→std)
[restart 0] nLML = -1961.196655
[restart 1] nLML = -1961.196655
[ok] Best nLML = -1961.196655
[fit] amp=2.842 | noise_var=0.0006804 | ell=[23.769  2.843 93.166  0.379]
[fit] train RMSE(z)=0.0155 | R²(z)=0.9998  (z = standardized)


In [25]:
TRAIN_CSV = "Surrogate_sensitivity_across_flowRates/P_flowrate_2_XY.csv" #Completed_train_set_10_30_25.csv#"Train_Set_2.csv"
TEST_CSV  = "Completed_Test_set_3.csv"#"Test_Set_2.csv"
CKPT_DIR  = "ckpts/gp_run_sens_04"
dim = 4
gp = MiniGPJax(
    dim=dim,
    kernel="rbf",            # or "matern32" I dont have the others
    fixed_noise=None,        # consider None to learn noise for better generalization
    standardize_x=True,
    standardize_y=True,
    log_y=True,              # ensures: log -> standardize (train stats) -> fit
    jitter=1e-6,
)
gp.load(TRAIN_CSV, TEST_CSV)
gp.fit(lbfgs_max_iter=30, lbfgs_tol=1e-7, num_restarts=2, seed=200)

[ok] Loaded train: X=(1024, 4), y=(1024,) (log→std)
[ok] Loaded test:  X=(256, 4), y=(256,) (log→std)
[restart 0] nLML = -1700.216675
[restart 1] nLML = -1700.216675
[ok] Best nLML = -1700.216675
[fit] amp=8.398 | noise_var=0.0009244 | ell=[15.624  6.167  7.471  1.035]
[fit] train RMSE(z)=0.0323 | R²(z)=0.9990  (z = standardized)


In [31]:
TRAIN_CSV = "Surrogate_sensitivity_across_flowRates/P_flowrate_3_XY.csv" #Completed_train_set_10_30_25.csv#"Train_Set_2.csv"
TEST_CSV  = "Completed_Test_set_3.csv"#"Test_Set_2.csv"
CKPT_DIR  = "ckpts/gp_run_sens_04"
dim = 4
gp = MiniGPJax(
    dim=dim,
    kernel="rbf",            # or "matern32" I dont have the others
    fixed_noise=None,        # consider None to learn noise for better generalization
    standardize_x=True,
    standardize_y=True,
    log_y=True,              # ensures: log -> standardize (train stats) -> fit
    jitter=1e-6,
)
gp.load(TRAIN_CSV, TEST_CSV)
gp.fit(lbfgs_max_iter=50, lbfgs_tol=1e-7, num_restarts=2, seed=500)

[ok] Loaded train: X=(1024, 4), y=(1024,) (log→std)
[ok] Loaded test:  X=(256, 4), y=(256,) (log→std)
[restart 0] nLML = 3326.518066
[restart 1] nLML = 3326.518066
[ok] Best nLML = 3326.518066
[fit] amp=6.157 | noise_var=72.79 | ell=[ 4.818 53.703 54.858  0.068]
[fit] train RMSE(z)=0.6366 | R²(z)=0.5947  (z = standardized)


In [27]:
TRAIN_CSV = "Surrogate_sensitivity_across_flowRates/P_flowrate_4_XY.csv" #Completed_train_set_10_30_25.csv#"Train_Set_2.csv"
TEST_CSV  = "Completed_Test_set_3.csv"#"Test_Set_2.csv"
CKPT_DIR  = "ckpts/gp_run_sens_04"
dim = 4
gp = MiniGPJax(
    dim=dim,
    kernel="rbf",            # or "matern32" I dont have the others
    fixed_noise=None,        # consider None to learn noise for better generalization
    standardize_x=True,
    standardize_y=True,
    log_y=True,              # ensures: log -> standardize (train stats) -> fit
    jitter=1e-6,
)
gp.load(TRAIN_CSV, TEST_CSV)
gp.fit(lbfgs_max_iter=30, lbfgs_tol=1e-7, num_restarts=2, seed=200)

[ok] Loaded train: X=(1024, 4), y=(1024,) (log→std)
[ok] Loaded test:  X=(256, 4), y=(256,) (log→std)
[restart 0] nLML = -2565.083252
[restart 1] nLML = -2565.083252
[ok] Best nLML = -2565.083252
[fit] amp=2.607 | noise_var=3.937e-05 | ell=[ 5.515  1.539 25.527  0.381]
[fit] train RMSE(z)=0.0052 | R²(z)=1.0000  (z = standardized)
